# 03C — Classificação dos compradores pela natureza jurídica (Receita Federal)

**Objetivo.** Identificar a natureza jurídica de cada comprador do BPS (pelo CNPJ),
usando o Cadastro Nacional da Pessoa Jurídica publicado pela Receita Federal,
na versão de **dezembro de 2023** (último mês do período analisado).

**Fonte.** Receita Federal do Brasil — Dados Abertos do CNPJ, pasta `2023-12`:
https://arquivos.receitafederal.gov.br/index.php/s/YggdBLfdninEJX9?dir=/2023-12
Arquivos usados: `Empresas0.zip` a `Empresas9.zip` e `Naturezas.zip`; na seção 8, `Estabelecimentos0` a `Estabelecimentos9` e `Municipios`.
Códigos de natureza jurídica conforme a Tabela de Natureza Jurídica (IBGE/Concla).

**O que esta rotina faz.** Monta a lista de todos os CNPJs compradores da base
harmonizada (2009–2023, inclusive os que a rotina 03B excluiu), procura cada um no
cadastro da Receita e gera tabelas de conferência.

**O que esta rotina NÃO faz.** Não altera nenhuma base. A regra que define o que é
"público" é uma decisão da autora e será aplicada depois, na rotina 03B.

**Entrada:** `Base Harmonizacao Campos`, `Base Harmonizacao Publica`, `Base Receita Federal/` (subpastas `CNPJ`, `Natureza`, `Estabelecimento`, `Municipio`)
**Saída:** `Base Classificacao Receita`

In [7]:
# ── CAMINHO DOS DADOS ────────────────────────────────────────────────────
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass   # fora do Colab: use a variável de ambiente BPS_DADOS
PASTA_DADOS = os.environ.get('BPS_DADOS', '/content/drive/MyDrive/TCC/Registro de Compras v2/Base v2')
# ─────────────────────────────────────────────────────────────────────────

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
!pip install polars==1.35.2 pandas==2.2.3 pyarrow --quiet

In [9]:
import zipfile
from pathlib import Path
import pandas as pd
import polars as pl
pl.Config.set_tbl_rows(-1); pl.Config.set_tbl_cols(-1); pl.Config.set_fmt_str_lengths(70)

BASE            = Path(PASTA_DADOS)
PASTA_HARM      = BASE / 'Base Harmonizacao Campos'
PASTA_PUBLICA   = BASE / 'Base Harmonizacao Publica'
# Receita Federal, dados abertos do CNPJ, pasta 2023-12 — uma subpasta por tipo de arquivo
PASTA_RF        = BASE / 'Base Receita Federal'
PASTA_EMPRESAS  = PASTA_RF / 'CNPJ'              # Empresas0 a Empresas9
PASTA_NATUREZA  = PASTA_RF / 'Natureza'          # Naturezas
PASTA_ESTAB     = PASTA_RF / 'Estabelecimento'   # Estabelecimentos0 a Estabelecimentos9 (seção 8)
PASTA_MUNIC     = PASTA_RF / 'Municipio'         # Municipios (seção 8)
PASTA_SAIDA     = BASE / 'Base Classificacao Receita'
ANO_INICIO, ANO_FIM = 2009, 2023
ARQUIVOS_EMPRESAS_ESPERADOS = 10      # Empresas0 a Empresas9
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

for p in [PASTA_HARM, PASTA_PUBLICA, PASTA_EMPRESAS, PASTA_NATUREZA, PASTA_ESTAB, PASTA_MUNIC]:
    print(f'{"OK " if p.exists() else "FALTANDO"}  {p}')

OK   /content/drive/MyDrive/TCC/Registro de Compras v2/Base v2/Base Harmonizacao Campos
FALTANDO  /content/drive/MyDrive/TCC/Registro de Compras v2/Base v2/Base Harmonizacao Publica
OK   /content/drive/MyDrive/TCC/Registro de Compras v2/Base v2/Base Receita Federal/CNPJ
OK   /content/drive/MyDrive/TCC/Registro de Compras v2/Base v2/Base Receita Federal/Natureza
OK   /content/drive/MyDrive/TCC/Registro de Compras v2/Base v2/Base Receita Federal/Estabelecimento
OK   /content/drive/MyDrive/TCC/Registro de Compras v2/Base v2/Base Receita Federal/Municipio


## 1. Arquivos da Receita encontrados
A rotina aceita os arquivos compactados (`.zip`) ou já descompactados.

In [10]:
def localizar(marca, pastas):
    """Arquivos cujo nome contém a marca (ex.: 'EMPRECSV'), soltos ou dentro de .zip, nas pastas
    indicadas e nas subpastas delas. Se o mesmo arquivo estiver em duas pastas, fica uma cópia só."""
    achados, nomes = [], set()
    for pasta in pastas:
        if not pasta.exists():
            continue
        for f in sorted(x for x in pasta.rglob('*') if x.is_file()):
            if f.suffix.lower() == '.zip':
                with zipfile.ZipFile(f) as z:
                    ok = any(marca in m.upper() for m in z.namelist())
            else:
                ok = marca in f.name.upper()
            if ok and f.name not in nomes:
                achados.append(f); nomes.add(f.name)
    return achados

ARQ_EMPRESAS  = localizar('EMPRECSV', [PASTA_EMPRESAS])
ARQ_NATUREZAS = localizar('NATJUCSV', [PASTA_NATUREZA, PASTA_EMPRESAS])
print('Empresas :', [str(f.relative_to(PASTA_RF)) for f in ARQ_EMPRESAS])
print('Naturezas:', [str(f.relative_to(PASTA_RF)) for f in ARQ_NATUREZAS])
assert len(ARQ_NATUREZAS) == 1, f'Esperado exatamente 1 arquivo de Naturezas em {PASTA_NATUREZA} ou {PASTA_EMPRESAS}; achados: {ARQ_NATUREZAS}'
if len(ARQ_EMPRESAS) != ARQUIVOS_EMPRESAS_ESPERADOS:
    print(f'\nATENCAO: {len(ARQ_EMPRESAS)} arquivo(s) de Empresas; o cadastro completo tem '
          f'{ARQUIVOS_EMPRESAS_ESPERADOS}. CNPJs que estiverem nos arquivos faltantes '
          f'aparecerao como NAO ENCONTRADOS.')

Empresas : ['CNPJ/Empresas0_K3241.K03200Y0.D31209.EMPRECSV', 'CNPJ/Empresas1_K3241.K03200Y1.D31209.EMPRECSV', 'CNPJ/Empresas2_K3241.K03200Y2.D31209.EMPRECSV', 'CNPJ/Empresas3_K3241.K03200Y3.D31209.EMPRECSV', 'CNPJ/Empresas4_K3241.K03200Y4.D31209.EMPRECSV', 'CNPJ/Empresas5_K3241.K03200Y5.D31209.EMPRECSV', 'CNPJ/Empresas6_K3241.K03200Y6.D31209.EMPRECSV', 'CNPJ/Empresas7_K3241.K03200Y7.D31209.EMPRECSV', 'CNPJ/Empresas8_K3241.K03200Y8.D31209.EMPRECSV', 'CNPJ/Empresas9_K3241.K03200Y9.D31209.EMPRECSV']
Naturezas: ['Natureza/Naturezas_F.K03200$Z.D31209.NATJUCSV']


## 2. Lista de compradores da base BPS
Todos os CNPJs de instituição compradora, de 2009 a 2023, com: nome mais frequente,
número de registros, primeiro e último ano, esfera informada pelo BPS (existe até 2016)
e se o CNPJ ficou na base pública da rotina 03B.

In [11]:
def dv_cnpj_ok(c):
    """Confere os dígitos verificadores do CNPJ (14 dígitos)."""
    if c is None or len(c) != 14 or not c.isdigit() or len(set(c)) == 1:
        return False
    def dv(base, pesos):
        s = sum(int(d) * p for d, p in zip(base, pesos)) % 11
        return '0' if s < 2 else str(11 - s)
    p1 = [5,4,3,2,9,8,7,6,5,4,3,2]
    p2 = [6] + p1
    return c[12] == dv(c[:12], p1) and c[13] == dv(c[:12] + c[12], p2)

CNPJ = (pl.col('cnpj_instituicao').cast(pl.Utf8).str.replace_all(r'\D', '')
          .str.pad_start(14, '0'))
fr = []
for ano in range(ANO_INICIO, ANO_FIM + 1):
    arq = PASTA_HARM / f'BPS_{ano}.parquet'
    cols = pl.read_parquet(arq, n_rows=0).columns
    df = pl.read_parquet(arq, columns=[c for c in ['cnpj_instituicao', 'instituicao', 'esfera'] if c in cols])
    if 'esfera' not in df.columns:
        df = df.with_columns(pl.lit(None, pl.Utf8).alias('esfera'))
    fr.append(df.with_columns(pl.lit(ano).alias('ano')))
bps = pl.concat(fr, how='diagonal').with_columns(
    pl.when(pl.col('cnpj_instituicao').cast(pl.Utf8).str.replace_all(r'\D', '').str.len_chars() > 0)
      .then(CNPJ).otherwise(None).alias('cnpj'),
    pl.col('instituicao').cast(pl.Utf8).str.replace_all('\u00a0', ' ').str.strip_chars().alias('instituicao'),
    pl.col('esfera').cast(pl.Utf8).str.replace_all('\u00a0', ' ').str.strip_chars().str.to_uppercase().alias('esfera'))

sem_cnpj = bps.filter(pl.col('cnpj').is_null())
print(f'Registros na base harmonizada {ANO_INICIO}-{ANO_FIM}: {bps.height:,}')
print(f'Registros SEM CNPJ do comprador (nao podem ser cruzados): {sem_cnpj.height:,}')

# A coluna 'na_base_publica_03B' é só conferência (a rotina 03B NÃO a usa). Numa execução do zero a 03B ainda não
# rodou, porque ela depende desta rotina; então a conferência fica vazia e é feita na próxima execução.
# (Correção de 17/10/2026: antes esta célula exigia a saída da 03B e impedia rodar o pipeline do zero.)
arqs_pub = [PASTA_PUBLICA / f'BPS_{ano}.parquet' for ano in range(ANO_INICIO, ANO_FIM + 1)]
TEM_03B = all(a.exists() for a in arqs_pub)
publicos = set()
if TEM_03B:
    for a in arqs_pub:
        d = pl.read_parquet(a, columns=['cnpj_instituicao'])
        publicos |= set(d.select(CNPJ.alias('c'))['c'].drop_nulls().to_list())
else:
    print('Base pública da 03B ainda não existe (execução do zero): conferência com a 03B fica para a próxima execução.')

compradores = (bps.filter(pl.col('cnpj').is_not_null())
    .group_by('cnpj').agg(
        pl.len().alias('registros'),
        pl.col('ano').min().alias('ano_min'), pl.col('ano').max().alias('ano_max'),
        pl.col('instituicao').drop_nulls().filter(pl.col('instituicao').drop_nulls() != '')
          .mode().first().alias('nome_bps'),
        pl.col('esfera').drop_nulls().filter(pl.col('esfera').drop_nulls() != '')
          .unique().sort().str.join('/').alias('esfera_bps'))
    .with_columns(pl.col('cnpj').str.slice(0, 8).alias('cnpj_basico'),
                  pl.col('cnpj').map_elements(dv_cnpj_ok, return_dtype=pl.Boolean).alias('cnpj_valido'),
                  (pl.col('cnpj').is_in(list(publicos)) if TEM_03B else pl.lit(None, pl.Boolean)).alias('na_base_publica_03B'),
                  pl.when(pl.col('esfera_bps') == '').then(None).otherwise(pl.col('esfera_bps')).alias('esfera_bps'))
    .sort('registros', descending=True))
print(f'CNPJs compradores distintos: {compradores.height:,}')
print(f'CNPJs com digito verificador invalido: {(~compradores["cnpj_valido"]).sum()}')

Registros na base harmonizada 2009-2023: 872,804
Registros SEM CNPJ do comprador (nao podem ser cruzados): 665
Base pública da 03B ainda não existe (execução do zero): conferência com a 03B fica para a próxima execução.
CNPJs compradores distintos: 2,081
CNPJs com digito verificador invalido: 0


## 3. Tabela de naturezas jurídicas (Receita)

In [12]:
def ler_receita(arq, marca, nomes, usecols=None, chunksize=None):
    """Le arquivo da Receita (latin-1, ';', sem cabecalho), solto ou dentro de .zip."""
    opts = dict(sep=';', header=None, names=nomes, dtype=str, encoding='latin-1',
                quotechar='"', usecols=usecols, chunksize=chunksize, keep_default_na=False)
    if arq.suffix.lower() == '.zip':
        z = zipfile.ZipFile(arq)
        membro = [m for m in z.namelist() if marca in m.upper()][0]
        return pd.read_csv(z.open(membro), **opts)
    return pd.read_csv(arq, **opts)

naturezas = pl.from_pandas(ler_receita(ARQ_NATUREZAS[0], 'NATJUCSV', ['natureza_cod', 'natureza_desc']))
GRUPOS = {'1': 'Administracao Publica', '2': 'Entidades Empresariais',
          '3': 'Entidades sem Fins Lucrativos', '4': 'Pessoas Fisicas',
          '5': 'Organizacoes Internacionais e Outras'}
naturezas = naturezas.with_columns(
    pl.col('natureza_cod').str.slice(0, 1).replace_strict(GRUPOS, default='Nao informado').alias('grupo_natureza'))
print(f'Naturezas juridicas na tabela: {naturezas.height}')

Naturezas juridicas na tabela: 90


## 4. Busca dos compradores no cadastro de Empresas
Lê cada arquivo em blocos e guarda só as linhas dos nossos CNPJs (pelos 8 primeiros
dígitos, o "CNPJ básico"). Pode levar alguns minutos por arquivo.

In [13]:
COLS_EMP = ['cnpj_basico', 'razao_social', 'natureza_cod', 'qualificacao_responsavel',
            'capital_social', 'porte', 'ente_federativo']
alvo = set(compradores['cnpj_basico'].to_list())
achados = []
for arq in ARQ_EMPRESAS:
    n_lidas, n_achadas = 0, 0
    for bloco in ler_receita(arq, 'EMPRECSV', COLS_EMP,
                             usecols=['cnpj_basico', 'razao_social', 'natureza_cod', 'porte', 'ente_federativo'],
                             chunksize=1_000_000):
        n_lidas += len(bloco)
        sel = bloco[bloco['cnpj_basico'].isin(alvo)]
        n_achadas += len(sel)
        if len(sel):
            achados.append(sel)
    print(f'{arq.name:<45} {n_lidas:>11,} linhas lidas | {n_achadas:>5} compradores encontrados')

receita = (pl.from_pandas(pd.concat(achados)) if achados
           else pl.DataFrame(schema={c: pl.Utf8 for c in ['cnpj_basico', 'razao_social', 'natureza_cod', 'porte', 'ente_federativo']}))
dup = receita.group_by('cnpj_basico').len().filter(pl.col('len') > 1).height
assert dup == 0, f'{dup} CNPJ basicos repetidos no cadastro da Receita'

Empresas0_K3241.K03200Y0.D31209.EMPRECSV       15,457,388 linhas lidas |   550 compradores encontrados
Empresas1_K3241.K03200Y1.D31209.EMPRECSV        4,494,860 linhas lidas |   222 compradores encontrados
Empresas2_K3241.K03200Y2.D31209.EMPRECSV        4,494,860 linhas lidas |   237 compradores encontrados
Empresas3_K3241.K03200Y3.D31209.EMPRECSV        4,494,860 linhas lidas |   714 compradores encontrados
Empresas4_K3241.K03200Y4.D31209.EMPRECSV        4,494,860 linhas lidas |   143 compradores encontrados
Empresas5_K3241.K03200Y5.D31209.EMPRECSV        4,494,860 linhas lidas |    62 compradores encontrados
Empresas6_K3241.K03200Y6.D31209.EMPRECSV        4,494,860 linhas lidas |    41 compradores encontrados
Empresas7_K3241.K03200Y7.D31209.EMPRECSV        4,494,860 linhas lidas |    29 compradores encontrados
Empresas8_K3241.K03200Y8.D31209.EMPRECSV        4,494,860 linhas lidas |    15 compradores encontrados
Empresas9_K3241.K03200Y9.D31209.EMPRECSV        4,494,860 linhas lidas | 

## 5. Cruzamento

In [14]:
cruz = (compradores
    .join(receita, on='cnpj_basico', how='left')
    .join(naturezas, on='natureza_cod', how='left')
    .with_columns(pl.col('natureza_cod').is_not_null().alias('encontrado_receita'),
                  pl.when(pl.col('ente_federativo') == '').then(None)
                    .otherwise(pl.col('ente_federativo')).alias('ente_federativo')))

DESTAQUE = {'2011': 'Empresa Publica', '2038': 'Sociedade de Economia Mista',
            '3301': 'Organizacao Social (OS)', '3077': 'Servico Social Autonomo',
            '2216': 'Empresa Domiciliada no Exterior'}
cruz = cruz.with_columns(pl.col('natureza_cod').replace_strict(DESTAQUE, default=None).alias('destaque'))

ordem = ['cnpj', 'cnpj_basico', 'cnpj_valido', 'nome_bps', 'razao_social', 'registros', 'ano_min', 'ano_max',
         'esfera_bps', 'na_base_publica_03B', 'encontrado_receita', 'natureza_cod', 'natureza_desc',
         'grupo_natureza', 'destaque', 'ente_federativo', 'porte']
cruz = cruz.select(ordem).sort('registros', descending=True)
cruz.write_parquet(PASTA_SAIDA / 'compradores_receita.parquet')
cruz.write_csv(PASTA_SAIDA / 'compradores_receita.csv')
print('Exportado: compradores_receita.csv / .parquet')

Exportado: compradores_receita.csv / .parquet


## 6. Conferências
### 6.1 Cobertura do cruzamento

In [15]:
cob = (cruz.group_by('encontrado_receita')
           .agg(pl.len().alias('cnpjs'), pl.col('registros').sum().alias('registros'))
           .with_columns((pl.col('registros') / pl.col('registros').sum() * 100).round(2).alias('pct_registros')))
print(cob)
print('\nCNPJs nao encontrados (maiores em registros):')
print(cruz.filter(~pl.col('encontrado_receita'))
          .select('cnpj', 'cnpj_valido', 'nome_bps', 'registros', 'ano_min', 'ano_max').head(20))

shape: (1, 4)
┌────────────────────┬───────┬───────────┬───────────────┐
│ encontrado_receita ┆ cnpjs ┆ registros ┆ pct_registros │
│ ---                ┆ ---   ┆ ---       ┆ ---           │
│ bool               ┆ u32   ┆ u32       ┆ f64           │
╞════════════════════╪═══════╪═══════════╪═══════════════╡
│ true               ┆ 2081  ┆ 872139    ┆ 100.0         │
└────────────────────┴───────┴───────────┴───────────────┘

CNPJs nao encontrados (maiores em registros):
shape: (0, 6)
┌──────┬─────────────┬──────────┬───────────┬─────────┬─────────┐
│ cnpj ┆ cnpj_valido ┆ nome_bps ┆ registros ┆ ano_min ┆ ano_max │
│ ---  ┆ ---         ┆ ---      ┆ ---       ┆ ---     ┆ ---     │
│ str  ┆ bool        ┆ str      ┆ u32       ┆ i32     ┆ i32     │
╞══════╪═════════════╪══════════╪═══════════╪═════════╪═════════╡
└──────┴─────────────┴──────────┴───────────┴─────────┴─────────┘


### 6.2 Distribuição por natureza jurídica

In [16]:
dist = (cruz.filter(pl.col('encontrado_receita'))
            .group_by('grupo_natureza', 'natureza_cod', 'natureza_desc')
            .agg(pl.len().alias('cnpjs'), pl.col('registros').sum().alias('registros'))
            .sort(['grupo_natureza', 'registros'], descending=[False, True]))
print(dist)
dist.write_csv(PASTA_SAIDA / 'distribuicao_natureza.csv')

shape: (21, 5)
┌───────────────────────────────┬──────────────┬───────────────────────────────┬───────┬───────────┐
│ grupo_natureza                ┆ natureza_cod ┆ natureza_desc                 ┆ cnpjs ┆ registros │
│ ---                           ┆ ---          ┆ ---                           ┆ ---   ┆ ---       │
│ str                           ┆ str          ┆ str                           ┆ u32   ┆ u32       │
╞═══════════════════════════════╪══════════════╪═══════════════════════════════╪═══════╪═══════════╡
│ Administracao Publica         ┆ 1244         ┆ Município                     ┆ 881   ┆ 385422    │
│ Administracao Publica         ┆ 1333         ┆ Fundo Público da              ┆ 890   ┆ 309905    │
│                               ┆              ┆ Administração Direta          ┆       ┆           │
│                               ┆              ┆ Municipal                     ┆       ┆           │
│ Administracao Publica         ┆ 1210         ┆ Consórcio Público de Direit

### 6.3 Conferência da esfera informada pelo BPS (2009–2016)
Compara a esfera que o próprio BPS atribui ao comprador com o grupo de natureza
jurídica da Receita. Contagem de CNPJs e de registros.

In [17]:
conf_esfera = (cruz.filter(pl.col('esfera_bps').is_not_null())
    .with_columns(pl.col('grupo_natureza').fill_null('NAO ENCONTRADO'))
    .group_by('esfera_bps', 'grupo_natureza')
    .agg(pl.len().alias('cnpjs'), pl.col('registros').sum().alias('registros'))
    .sort(['esfera_bps', 'registros'], descending=[False, True]))
print(conf_esfera)
conf_esfera.write_csv(PASTA_SAIDA / 'conferencia_esfera_bps.csv')

shape: (8, 4)
┌──────────────────┬───────────────────────────────┬───────┬───────────┐
│ esfera_bps       ┆ grupo_natureza                ┆ cnpjs ┆ registros │
│ ---              ┆ ---                           ┆ ---   ┆ ---       │
│ str              ┆ str                           ┆ u32   ┆ u32       │
╞══════════════════╪═══════════════════════════════╪═══════╪═══════════╡
│ ESTADUAL         ┆ Administracao Publica         ┆ 64    ┆ 43504     │
│ ESTADUAL         ┆ Entidades Empresariais        ┆ 1     ┆ 1         │
│ ESTADUAL/PRIVADA ┆ Entidades sem Fins Lucrativos ┆ 1     ┆ 1651      │
│ FEDERAL          ┆ Administracao Publica         ┆ 19    ┆ 6137      │
│ MUNICIPAL        ┆ Administracao Publica         ┆ 883   ┆ 417829    │
│ PRIVADA          ┆ Entidades sem Fins Lucrativos ┆ 45    ┆ 32433     │
│ PRIVADA          ┆ Administracao Publica         ┆ 5     ┆ 15        │
│ PRIVADA          ┆ Entidades Empresariais        ┆ 1     ┆ 2         │
└──────────────────┴─────────────────

### 6.4 Conferência da classificação atual (rotina 03B)

In [18]:
if not TEM_03B:
    print('Conferência com a 03B pulada: a base pública ainda não existe (rode a 03B e depois esta rotina de novo, se quiser a conferência).')
conf_03b = (cruz.filter(pl.lit(TEM_03B)).with_columns(pl.col('grupo_natureza').fill_null('NAO ENCONTRADO'),
                              pl.when(pl.col('na_base_publica_03B')).then(pl.lit('mantido como publico'))
                                .otherwise(pl.lit('excluido')).alias('situacao_03B'))
    .group_by('situacao_03B', 'grupo_natureza')
    .agg(pl.len().alias('cnpjs'), pl.col('registros').sum().alias('registros'))
    .sort(['situacao_03B', 'registros'], descending=[False, True]))
print(conf_03b)
conf_03b.write_csv(PASTA_SAIDA / 'conferencia_03B.csv')

Conferência com a 03B pulada: a base pública ainda não existe (rode a 03B e depois esta rotina de novo, se quiser a conferência).
shape: (0, 4)
┌──────────────┬────────────────┬───────┬───────────┐
│ situacao_03B ┆ grupo_natureza ┆ cnpjs ┆ registros │
│ ---          ┆ ---            ┆ ---   ┆ ---       │
│ str          ┆ str            ┆ u32   ┆ u32       │
╞══════════════╪════════════════╪═══════╪═══════════╡
└──────────────┴────────────────┴───────┴───────────┘


### 6.5 Naturezas de fronteira (decisão da autora)
Empresa pública, sociedade de economia mista, OS, serviço social autônomo e empresa
domiciliada no exterior.

In [19]:
print(cruz.filter(pl.col('destaque').is_not_null())
          .select('destaque', 'cnpj', 'nome_bps', 'razao_social', 'registros', 'na_base_publica_03B')
          .sort(['destaque', 'registros'], descending=[False, True]))

shape: (3, 6)
┌─────────────────┬────────────────┬─────────────────┬────────────────┬───────────┬────────────────┐
│ destaque        ┆ cnpj           ┆ nome_bps        ┆ razao_social   ┆ registros ┆ na_base_public │
│ ---             ┆ ---            ┆ ---             ┆ ---            ┆ ---       ┆ a_03B          │
│ str             ┆ str            ┆ str             ┆ str            ┆ u32       ┆ ---            │
│                 ┆                ┆                 ┆                ┆           ┆ bool           │
╞═════════════════╪════════════════╪═════════════════╪════════════════╪═══════════╪════════════════╡
│ Empresa Publica ┆ 15126437002278 ┆ EMPRESA         ┆ EMPRESA        ┆ 1         ┆ null           │
│                 ┆                ┆ BRASILEIRA DE   ┆ BRASILEIRA DE  ┆           ┆                │
│                 ┆                ┆ SERVICOS        ┆ SERVICOS       ┆           ┆                │
│                 ┆                ┆ HOSPITALARES -  ┆ HOSPITALARES - ┆      

### 6.6 Os 13 compradores sem classificação a partir de 2017

In [20]:
arq_rev = PASTA_PUBLICA / 'casos_revisao_manual.csv'
if arq_rev.exists():
    rev = pl.read_csv(arq_rev, infer_schema_length=0).with_columns(
        pl.col('cnpj').str.replace_all(r'\D', '').str.pad_start(14, '0'))
    print(rev.select('cnpj').join(cruz, on='cnpj', how='left')
             .select('cnpj', 'nome_bps', 'razao_social', 'natureza_cod', 'natureza_desc', 'ente_federativo', 'registros'))
else:
    print('casos_revisao_manual.csv nao encontrado')

casos_revisao_manual.csv nao encontrado


## 7. Memória de cálculo

In [21]:
print(f'Fonte: Receita Federal, dados abertos do CNPJ, pasta 2023-12 '
      f'({len(ARQ_EMPRESAS)} de {ARQUIVOS_EMPRESAS_ESPERADOS} arquivos de Empresas)')
print(f'Registros na base harmonizada {ANO_INICIO}-{ANO_FIM}  : {bps.height:,}')
print(f'  sem CNPJ do comprador (fora do cruzamento): {sem_cnpj.height:,}')
print(f'CNPJs compradores distintos                 : {compradores.height:,}')
print(f'  com digito verificador invalido           : {(~cruz["cnpj_valido"]).sum():,}')
print(f'  encontrados na Receita                    : {cruz["encontrado_receita"].sum():,}')
print(f'  nao encontrados                           : {(~cruz["encontrado_receita"]).sum():,}')
print('\nArquivos gerados em', PASTA_SAIDA)
for f in sorted(PASTA_SAIDA.iterdir()):
    print('  ', f.name)

Fonte: Receita Federal, dados abertos do CNPJ, pasta 2023-12 (10 de 10 arquivos de Empresas)
Registros na base harmonizada 2009-2023  : 872,804
  sem CNPJ do comprador (fora do cruzamento): 665
CNPJs compradores distintos                 : 2,081
  com digito verificador invalido           : 0
  encontrados na Receita                    : 2,081
  nao encontrados                           : 0

Arquivos gerados em /content/drive/MyDrive/TCC/Registro de Compras v2/Base v2/Base Classificacao Receita
   compradores_receita.csv
   compradores_receita.parquet
   conferencia_03B.csv
   conferencia_esfera_bps.csv
   distribuicao_natureza.csv


## 8. Conferência da UF do comprador: BPS × Receita (Estabelecimentos, dez/2023)
**Somente mede; não altera nenhuma base.** Compara a UF que o próprio BPS informa para cada comprador (sem a UF recuperada pela rotina 03) com a UF do estabelecimento no cadastro da Receita de dez/2023. Serve para decidir, no registro de decisões, se a Receita pode preencher a UF de compras que ficaram sem UF (caso da Fundação de Hematologia e Hemoterapia da Bahia, 110 compras em 2011–2012).

**Fonte.** Receita Federal — Dados Abertos do CNPJ, pasta `2023-12`, arquivos `Estabelecimentos0` a `Estabelecimentos9` e `Municipios`. Campo UF conforme o documento *CNPJ – Metadados* da Receita: "sigla da unidade da federação em que se encontra o estabelecimento".

**Limite.** O arquivo mostra o endereço vigente em dez/2023, não o da data da compra.

**Entrada:** `Base Receita Federal/Estabelecimento` e `Base Receita Federal/Municipio` · **Saída:** `Base Classificacao Receita/conferencia_uf_receita`

In [22]:
# SOMENTE MEDE. Não altera nenhuma base. Serve para decidir (registro de decisões) se a UF
# da Receita pode preencher compras que ficaram sem UF depois da recuperação da rotina 03.
# Fonte: Receita Federal, dados abertos do CNPJ, pasta 2023-12, arquivos Estabelecimentos0-9
# (e Municipios, na pasta Municipio). Layout conforme o documento "CNPJ - Metadados" da Receita:
# UF = "sigla da unidade da federação em que se encontra o estabelecimento".
# Atenção: o arquivo mostra o endereço vigente em dez/2023, não o da data da compra.

ARQUIVOS_ESTAB_ESPERADOS = 10      # Estabelecimentos0 a Estabelecimentos9
ARQ_ESTAB = localizar('ESTABELE', [PASTA_ESTAB])
ARQ_MUNIC = localizar('MUNICCSV', [PASTA_MUNIC, PASTA_ESTAB, PASTA_EMPRESAS])
print('Estabelecimentos:', [str(f.relative_to(PASTA_RF)) for f in ARQ_ESTAB])
print('Municipios      :', [str(f.relative_to(PASTA_RF)) for f in ARQ_MUNIC] or 'não encontrado (o município sai só com o código da Receita)')
assert ARQ_ESTAB, f'Nenhum arquivo de Estabelecimentos em {PASTA_ESTAB}'
if len(ARQ_ESTAB) != ARQUIVOS_ESTAB_ESPERADOS:
    print(f'\nATENCAO: {len(ARQ_ESTAB)} arquivo(s) de Estabelecimentos; o cadastro completo tem '
          f'{ARQUIVOS_ESTAB_ESPERADOS}. CNPJs dos arquivos faltantes aparecerão como NAO ENCONTRADOS.')


Estabelecimentos: ['Estabelecimento/K3241.K03200Y0.D31209.ESTABELE', 'Estabelecimento/K3241.K03200Y1.D31209.ESTABELE', 'Estabelecimento/K3241.K03200Y2.D31209.ESTABELE', 'Estabelecimento/K3241.K03200Y3.D31209.ESTABELE', 'Estabelecimento/K3241.K03200Y4.D31209.ESTABELE', 'Estabelecimento/K3241.K03200Y5.D31209.ESTABELE', 'Estabelecimento/K3241.K03200Y6.D31209.ESTABELE', 'Estabelecimento/K3241.K03200Y7.D31209.ESTABELE', 'Estabelecimento/K3241.K03200Y8.D31209.ESTABELE', 'Estabelecimento/K3241.K03200Y9.D31209.ESTABELE']
Municipios      : ['Municipio/F.K03200$Z.D31209.MUNICCSV']


### 8.1 UF de cada comprador segundo o próprio BPS (só a UF que veio na fonte)

In [23]:
fr = []
for ano in range(ANO_INICIO, ANO_FIM + 1):
    arq = PASTA_HARM / f'BPS_{ano}.parquet'
    cols = pl.read_parquet(arq, n_rows=0).columns
    df = pl.read_parquet(arq, columns=[c for c in ['cnpj_instituicao', 'uf', 'local_recuperado'] if c in cols])
    if 'local_recuperado' not in df.columns:
        df = df.with_columns(pl.lit(False).alias('local_recuperado'))
    fr.append(df.with_columns(pl.lit(ano).alias('ano')))
uf_txt = pl.col('uf').cast(pl.Utf8).str.strip_chars().str.to_uppercase()
bps_uf = (pl.concat(fr, how='diagonal')
            .filter(pl.col('cnpj_instituicao').cast(pl.Utf8).str.replace_all(r'\D', '').str.len_chars() > 0)
            .with_columns(CNPJ.alias('cnpj'),
                          pl.when((uf_txt.str.len_chars() > 0) & ~pl.col('local_recuperado').fill_null(False))
                            .then(uf_txt).otherwise(None).alias('uf_fonte')))
uf_por_cnpj = (bps_uf.group_by('cnpj').agg(
                   pl.col('uf_fonte').drop_nulls().unique().sort().str.join('/').alias('uf_bps'),
                   pl.col('uf_fonte').drop_nulls().n_unique().alias('n_uf_bps'),
                   pl.col('uf_fonte').is_null().sum().alias('registros_sem_uf_fonte'))
               .with_columns(pl.when(pl.col('uf_bps') == '').then(None).otherwise(pl.col('uf_bps')).alias('uf_bps')))
print(f'Compradores com UF na fonte  : {uf_por_cnpj.filter(pl.col("n_uf_bps") >= 1).height:,}')
print(f'  com mais de uma UF na fonte: {uf_por_cnpj.filter(pl.col("n_uf_bps") > 1).height:,}')
print(f'Compradores NUNCA com UF na fonte: {uf_por_cnpj.filter(pl.col("n_uf_bps") == 0).height:,}')


Compradores com UF na fonte  : 2,080
  com mais de uma UF na fonte: 0
Compradores NUNCA com UF na fonte: 1


### 8.2 Busca dos compradores no arquivo de Estabelecimentos (CNPJ completo, 14 dígitos)

In [24]:
COLS_ESTAB = ['cnpj_basico', 'cnpj_ordem', 'cnpj_dv', 'matriz_filial', 'nome_fantasia', 'situacao_cadastral',
              'data_situacao_cadastral', 'motivo_situacao', 'cidade_exterior', 'pais', 'data_inicio_atividade',
              'cnae_principal', 'cnae_secundaria', 'tipo_logradouro', 'logradouro', 'numero', 'complemento',
              'bairro', 'cep', 'uf', 'municipio_cod', 'ddd1', 'telefone1', 'ddd2', 'telefone2', 'ddd_fax', 'fax',
              'email', 'situacao_especial', 'data_situacao_especial']
USAR = ['cnpj_basico', 'cnpj_ordem', 'cnpj_dv', 'matriz_filial', 'situacao_cadastral', 'data_situacao_cadastral',
        'data_inicio_atividade', 'uf', 'municipio_cod']
alvo_basico = set(compradores['cnpj_basico'].to_list())
alvo_cnpj = set(compradores['cnpj'].to_list())
achados_est = []
for arq in ARQ_ESTAB:
    n_lidas, n_achadas = 0, 0
    for bloco in ler_receita(arq, 'ESTABELE', COLS_ESTAB, usecols=USAR, chunksize=1_000_000):
        n_lidas += len(bloco)
        sel = bloco[bloco['cnpj_basico'].isin(alvo_basico)]
        sel = sel[(sel['cnpj_basico'] + sel['cnpj_ordem'] + sel['cnpj_dv']).isin(alvo_cnpj)]
        n_achadas += len(sel)
        if len(sel):
            achados_est.append(sel)
    print(f'{arq.name:<45} {n_lidas:>11,} linhas lidas | {n_achadas:>5} compradores encontrados')

estab = (pl.from_pandas(pd.concat(achados_est)) if achados_est else pl.DataFrame(schema={c: pl.Utf8 for c in USAR}))
estab = estab.with_columns((pl.col('cnpj_basico') + pl.col('cnpj_ordem') + pl.col('cnpj_dv')).alias('cnpj'),
                           pl.col('uf').str.strip_chars().str.to_uppercase().alias('uf_receita'))
dup = estab.group_by('cnpj').len().filter(pl.col('len') > 1).height
assert dup == 0, f'{dup} CNPJs repetidos no arquivo de Estabelecimentos'

SITUACAO = {'01': 'NULA', '1': 'NULA', '02': 'ATIVA', '2': 'ATIVA', '03': 'SUSPENSA', '3': 'SUSPENSA',
            '04': 'INAPTA', '4': 'INAPTA', '08': 'BAIXADA', '8': 'BAIXADA'}
estab = estab.with_columns(pl.col('situacao_cadastral').replace_strict(SITUACAO, default='OUTRA').alias('situacao'),
                           pl.col('data_inicio_atividade').str.strptime(pl.Date, '%Y%m%d', strict=False).alias('inicio_atividade'),
                           pl.col('data_situacao_cadastral').str.strptime(pl.Date, '%Y%m%d', strict=False).alias('data_situacao'))
if ARQ_MUNIC:
    munic = pl.from_pandas(ler_receita(ARQ_MUNIC[0], 'MUNICCSV', ['municipio_cod', 'municipio_receita']))
    estab = estab.join(munic, on='municipio_cod', how='left')
else:
    estab = estab.with_columns(pl.lit(None, pl.Utf8).alias('municipio_receita'))


K3241.K03200Y0.D31209.ESTABELE                 16,053,486 linhas lidas |   519 compradores encontrados
K3241.K03200Y1.D31209.ESTABELE                  4,753,435 linhas lidas |   214 compradores encontrados
K3241.K03200Y2.D31209.ESTABELE                  4,753,435 linhas lidas |   192 compradores encontrados
K3241.K03200Y3.D31209.ESTABELE                  4,753,435 linhas lidas |   204 compradores encontrados
K3241.K03200Y4.D31209.ESTABELE                  4,753,435 linhas lidas |   114 compradores encontrados
K3241.K03200Y5.D31209.ESTABELE                  4,753,435 linhas lidas |   121 compradores encontrados
K3241.K03200Y6.D31209.ESTABELE                  4,753,435 linhas lidas |   178 compradores encontrados
K3241.K03200Y7.D31209.ESTABELE                  4,753,435 linhas lidas |   195 compradores encontrados
K3241.K03200Y8.D31209.ESTABELE                  4,753,435 linhas lidas |   210 compradores encontrados
K3241.K03200Y9.D31209.ESTABELE                  4,753,435 linhas lidas | 

### 8.3 Validação: a UF da Receita bate com a UF informada pelo BPS?

In [25]:
conf_uf = (compradores.select('cnpj', 'nome_bps', 'registros', 'ano_min', 'ano_max', 'esfera_bps', 'na_base_publica_03B')
    .join(uf_por_cnpj, on='cnpj', how='left')
    .join(estab.select('cnpj', 'uf_receita', 'municipio_cod', 'municipio_receita', 'situacao', 'data_situacao',
                       'inicio_atividade', 'matriz_filial'), on='cnpj', how='left')
    .with_columns(pl.col('uf_receita').is_not_null().alias('encontrado_estab'),
                  pl.when(pl.col('uf_bps').is_null()).then(pl.lit('BPS sem UF'))
                    .when(pl.col('uf_receita').is_null()).then(pl.lit('nao encontrado na Receita'))
                    .when(pl.col('n_uf_bps') > 1).then(pl.lit('BPS com mais de uma UF'))
                    .when(pl.col('uf_bps') == pl.col('uf_receita')).then(pl.lit('igual'))
                    .otherwise(pl.lit('diferente')).alias('comparacao'))
    .sort('registros', descending=True))

for nome, filtro in [('Todos os compradores', pl.lit(True))] + ([('Base pública (03B)', pl.col('na_base_publica_03B'))] if TEM_03B else []):
    t = (conf_uf.filter(filtro).group_by('comparacao')
            .agg(pl.len().alias('cnpjs'), pl.col('registros').sum().alias('registros')).sort('cnpjs', descending=True))
    print(f'\n{nome}'); print(t)
    comp = conf_uf.filter(filtro & pl.col('comparacao').is_in(['igual', 'diferente']))
    if comp.height:
        ig = comp.filter(pl.col('comparacao') == 'igual')
        print(f'  VALIDACAO: UF igual em {ig.height:,} de {comp.height:,} compradores = {ig.height / comp.height:.2%}'
              f' | em registros: {ig["registros"].sum() / comp["registros"].sum():.2%}')

print('\nCompradores com UF DIFERENTE (maiores em registros):')
print(conf_uf.filter(pl.col('comparacao') == 'diferente')
             .select('cnpj', 'nome_bps', 'registros', 'ano_min', 'ano_max', 'uf_bps', 'uf_receita', 'municipio_receita',
                     'situacao', 'data_situacao', 'inicio_atividade').head(30))

print('\nCompradores que NUNCA trazem UF no BPS (o que a Receita diz):')
print(conf_uf.filter(pl.col('comparacao') == 'BPS sem UF')
             .select('cnpj', 'nome_bps', 'registros', 'ano_min', 'ano_max', 'esfera_bps', 'uf_receita',
                     'municipio_receita', 'situacao', 'data_situacao', 'inicio_atividade', 'na_base_publica_03B'))

conf_uf.write_parquet(PASTA_SAIDA / 'conferencia_uf_receita.parquet')
conf_uf.write_csv(PASTA_SAIDA / 'conferencia_uf_receita.csv')
print('\nExportado: Base Classificacao Receita/conferencia_uf_receita.csv / .parquet (nenhuma base de análise foi alterada)')



Todos os compradores
shape: (2, 3)
┌────────────┬───────┬───────────┐
│ comparacao ┆ cnpjs ┆ registros │
│ ---        ┆ ---   ┆ ---       │
│ str        ┆ u32   ┆ u32       │
╞════════════╪═══════╪═══════════╡
│ igual      ┆ 2080  ┆ 872029    │
│ BPS sem UF ┆ 1     ┆ 110       │
└────────────┴───────┴───────────┘
  VALIDACAO: UF igual em 2,080 de 2,080 compradores = 100.00% | em registros: 100.00%

Compradores com UF DIFERENTE (maiores em registros):
shape: (0, 11)
┌──────┬─────────┬─────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┐
│ cnpj ┆ nome_bp ┆ registr ┆ ano_mi ┆ ano_ma ┆ uf_bps ┆ uf_rec ┆ munici ┆ situac ┆ data_s ┆ inicio │
│ ---  ┆ s       ┆ os      ┆ n      ┆ x      ┆ ---    ┆ eita   ┆ pio_re ┆ ao     ┆ ituaca ┆ _ativi │
│ str  ┆ ---     ┆ ---     ┆ ---    ┆ ---    ┆ str    ┆ ---    ┆ ceita  ┆ ---    ┆ o      ┆ dade   │
│      ┆ str     ┆ u32     ┆ i32    ┆ i32    ┆        ┆ str    ┆ ---    ┆ str    ┆ ---    ┆ ---    │
│      ┆         ┆      